<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# 模块 4.4：一个 FIRRTL 转换示例

**上一步：[常用 Pass 惯用法](4.3_firrtl_common_idioms.ipynb)**<br>

此 AnalyzeCircuit 转换遍历 `firrtl.ir.Circuit`，并记录它找到的每个模块的加法操作数。

## 设置

请运行以下命令：

In [ ]:
val path = System.getProperty("user.dir") + "/source/load-ivy.sc"
interp.load.module(ammonite.ops.Path(java.nio.file.FileSystems.getDefault().getPath(path)))

In [ ]:
// 编译器基础设施

// Firrtl IR 类

// Map 函数

// Scala 的可变集合
import scala.collection.mutable



## 计算每个模块的加法器数量

如前所述，Firrtl 电路使用树形表示：
  - Firrtl `Circuit` 包含一个 `DefModule` 序列。
  - `DefModule` 包含一个 `Port` 序列，可能还有一个 `Statement`。
  - `Statement` 可以包含其他 `Statement` 或 `Expression`。
  - `Expression` 可以包含其他 `Expression`。

要访问电路中的所有 Firrtl IR 节点，我们编写递归遍历此树的函数。为了记录统计信息，我们将传递一个 `Ledger` 类，并在遇到加法运算时使用它：

In [ ]:
class Ledger {
  import firrtl.Utils
  private var moduleName: Option[String] = None
  private val modules = mutable.Set[String]()
  private val moduleAddMap = mutable.Map[String, Int]()
  def foundAdd(): Unit = moduleName match {
    case None => sys.error("Ledger 中未定义模块名称！")
    case Some(name) => moduleAddMap(name) = moduleAddMap.getOrElse(name, 0) + 1
  }
  def getModuleName: String = moduleName match {
    case None => Utils.error("Ledger 中未定义模块名称！")
    case Some(name) => name
  }
  def setModuleName(myName: String): Unit = {
    modules += myName
    moduleName = Some(myName)
  }
  def serialize: String = {
    modules map { myName =>
      s"$myName => ${moduleAddMap.getOrElse(myName, 0)} 个加法操作！"
    } mkString "\n"
  }
}

现在，让我们定义一个 FIRRTL 转换，它遍历电路并在遇到加法器（带有 op 参数 `Add` 的 `DoPrim`）时更新我们的 `Ledger`。暂时不用担心 `inputForm` 或 `outputForm`。

花一些时间来理解 `walkModule`、`walkStatement` 和 `walkExpression` 如何遍历 FIRRTL AST 中的所有 `DefModule`、`Statement` 和 `Expression` 节点。

要回答的问题：
  - **为什么 walkModule 不调用 walkExpression？**
  - **为什么 walkExpression 执行后序遍历？**
  - **您能修改 walkExpression 以对表达式执行先序遍历吗？**

In [ ]:
class AnalyzeCircuit extends firrtl.Transform {
  import firrtl._
  import firrtl.ir._
  import firrtl.Mappers._
  import firrtl.Parser._
  import firrtl.annotations._
  import firrtl.PrimOps._
    
  // 要求 [[Circuit]] 形式为“low”
  def inputForm = LowForm
  // 指示输出 [[Circuit]] 形式为“low”
  def outputForm = LowForm

  // 由 [[Compiler]] 调用以运行您的过程。[[CircuitState]] 包含
  // 电路及其形式，以及其他相关数据。
  def execute(state: CircuitState): CircuitState = {
    val ledger = new Ledger()
    val circuit = state.circuit

    // 在电路中的每个 [[DefModule]] 上执行函数 walkModule(ledger)，
    // 返回一个带有新的 [[Seq]] of [[DefModule]] 的新 [[Circuit]]。
    //   - “高阶函数” - 将函数用作对象
    //   - “函数柯里化” - 部分参数表示法
    //   - “中缀表示法” - 花哨的函数调用语法
    //   - “map” - 经典的函数式编程概念
    //   - 丢弃返回的新 [[Circuit]]，因为电路未修改
    circuit map walkModule(ledger)

    // 打印我们的账本
    println(ledger.serialize)

    // 返回未更改的 [[CircuitState]]
    state
  }

  // 深度访问 m 中的每个 [[Statement]]。
  def walkModule(ledger: Ledger)(m: DefModule): DefModule = {
    // 将账本设置为当前模块名称
    ledger.setModuleName(m.name)

    // 在 m 中的每个 [[Statement]] 上执行函数 walkStatement(ledger)。
    //   - 返回新的 [[DefModule]]（在本例中，它与 m 相同）
    //   - 如果 m 不包含 [[Statement]]，则 map 返回 m。
    m map walkStatement(ledger)
  }

  // 深度访问 s 中的每个 [[Statement]] 和 [[Expression]]。
  def walkStatement(ledger: Ledger)(s: Statement): Statement = {

    // 在 s 中的每个 [[Expression]] 上执行函数 walkExpression(ledger)。
    //   - 丢弃新的 [[Statement]]（在本例中，它与 s 相同）
    //   - 如果 s 不包含 [[Expression]]，则 map 返回 s。
    s map walkExpression(ledger)

    // 在 s 中的每个 [[Statement]] 上执行函数 walkStatement(ledger)。
    //   - 返回新的 [[Statement]]（在本例中，它与 s 相同）
    //   - 如果 s 不包含 [[Statement]]，则 map 返回 s。
    s map walkStatement(ledger)
  }

  // 深度访问 e 中的每个 [[Expression]]。
  //   - “后序遍历” - 在处理 e 之前处理 e 的子 [[Expression]]
  def walkExpression(ledger: Ledger)(e: Expression): Expression = {

    // 在 e 中的每个 [[Expression]] 上执行函数 walkExpression(ledger)。
    //   - 返回新的 [[Expression]]（在本例中，它与 e 相同）
    //   - 如果 s 不包含 [[Expression]]，则 map 返回 e。
    val visited = e map walkExpression(ledger)

    visited match {
      // 如果 e 是加法器，则递增我们的账本并返回 e。
      case DoPrim(Add, _, _, _) =>
        ledger.foundAdd
        e
      // 如果 e 不是加法器，则返回 e。
      case notadd => notadd
    }
  }
}

## 运行我们的转换

既然我们已经定义了它，让我们在一个 Chisel 设计上运行它！首先，让我们定义一个 Chisel 模块。

In [ ]:
// Chisel 相关
import chisel3._
import chisel3.Input // 技术性：避免与 _root_.almond.input.Input 冲突
import chisel3.util._

In [ ]:
class AddMe(val nInputs: Int, val width: Int) extends Module {
  val io = IO(new Bundle {
    val in  = Input(Vec(nInputs, UInt(width.W)))
    val out = Output(UInt(width.W))
  })
  io.out := io.in.reduce(_ +& _)
}

接下来，让我们将其细化为 FIRRTL AST 语法。

In [ ]:
val firrtlSerialization = chisel3.Driver.emit(() => new AddMe(8, 4))

最后，让我们将 FIRRTL 编译为 Verilog，但在编译中包含我们的自定义转换。请注意，它会打印出找到的加法操作的数量！

**注意**（2021 年 1 月）：由于一个 [bug](https://github.com/freechipsproject/chisel-bootcamp/issues/129)，以下行可能已损坏。

In [ ]:
val verilog = compileFIRRTL(firrtlSerialization, new firrtl.VerilogCompiler(), Seq(new AnalyzeCircuit()))

`compileFIRRTL` 函数仅在本教程中定义——在后续章节中，我们将描述插入自定义转换的过程。

本节到此结束！